In [ ]:
from pathlib import Path

import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ARTIFACT_DIR = Path("artifacts")

INPUT_PATH = ARTIFACT_DIR / "01_ml_table.csv"
OUTPUT_PATH = ARTIFACT_DIR / "02_labeled_table.csv"


# ============================================================
# 2. VERIFY INPUT ARTIFACT
# ============================================================

assert INPUT_PATH.exists(), (
    f"Input artifact not found: {INPUT_PATH}"
)

print(f"Reading artifact: {INPUT_PATH}")


# ============================================================
# 3. LOAD ML TABLE
# ============================================================

df = pd.read_csv(
    INPUT_PATH,
    low_memory=False,
)

print("\nINPUT DATA")
print("=" * 80)

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")

assert df["order_id"].is_unique, (
    "order_id is not unique."
)


# ============================================================
# 4. PARSE DATE COLUMNS
# ============================================================

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        format="mixed",
        errors="coerce",
    )


# ============================================================
# 5. CHECK ORDER STATUS DISTRIBUTION
# ============================================================

print("\nORDER STATUS DISTRIBUTION")
print("=" * 80)

status_counts = (
    df["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="count")
)

status_counts["percentage"] = (
    status_counts["count"]
    / len(df)
    * 100
)

display(status_counts)


# ============================================================
# 6. KEEP ORDERS WITH A KNOWN DELIVERY OUTCOME
# ============================================================
#
# The target is whether an order was delivered late or on time.
#
# For training this target, we need:
#   1. A delivered order
#   2. Actual customer delivery date
#   3. Estimated delivery date
#
# Canceled / shipped / unavailable orders do not provide
# the same completed delivery outcome.
# ============================================================

eligible_mask = (
    df["order_status"].eq("delivered")
    & df["order_delivered_customer_date"].notna()
    & df["order_estimated_delivery_date"].notna()
)

labeled_df = (
    df.loc[eligible_mask]
    .copy()
    .reset_index(drop=True)
)

excluded_count = len(df) - len(labeled_df)

print("\nLABEL ELIGIBILITY")
print("=" * 80)

print(f"Original orders: {len(df):,}")
print(f"Eligible orders: {len(labeled_df):,}")
print(f"Excluded orders: {excluded_count:,}")


# ============================================================
# 7. CREATE LABEL
# ============================================================
#
# Compare CALENDAR DATES, not full timestamps.
#
# Example:
# actual delivery:    2018-08-13 14:00
# estimated delivery: 2018-08-13 00:00
#
# This is still ON TIME because both occurred on the same
# calendar day.
#
# is_late:
#   0 = on time or early
#   1 = late
# ============================================================

actual_delivery_date = (
    labeled_df["order_delivered_customer_date"]
    .dt.normalize()
)

estimated_delivery_date = (
    labeled_df["order_estimated_delivery_date"]
    .dt.normalize()
)

labeled_df["is_late"] = (
    actual_delivery_date
    > estimated_delivery_date
).astype("int8")


# ============================================================
# 8. CALCULATE DELIVERY DIFFERENCE FOR VALIDATION
# ============================================================
#
# This column is useful for checking the label.
# It is NOT a prediction-time feature.
# ============================================================

labeled_df["delivery_delay_days"] = (
    actual_delivery_date
    - estimated_delivery_date
).dt.days


# ============================================================
# 9. VALIDATE LABEL LOGIC
# ============================================================

assert labeled_df["is_late"].isin([0, 1]).all()

assert (
    labeled_df.loc[
        labeled_df["delivery_delay_days"] > 0,
        "is_late",
    ]
    == 1
).all()

assert (
    labeled_df.loc[
        labeled_df["delivery_delay_days"] <= 0,
        "is_late",
    ]
    == 0
).all()

print("\nLABEL LOGIC VALIDATION: PASS")


# ============================================================
# 10. MANUAL CHECK ON REAL ORDERS
# ============================================================

validation_columns = [
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay_days",
    "is_late",
]

late_examples = (
    labeled_df[
        labeled_df["is_late"] == 1
    ][validation_columns]
    .head(5)
)

on_time_examples = (
    labeled_df[
        labeled_df["is_late"] == 0
    ][validation_columns]
    .head(5)
)

print("\nREAL LATE ORDERS")
print("=" * 80)

display(late_examples)

print("\nREAL ON-TIME / EARLY ORDERS")
print("=" * 80)

display(on_time_examples)


# ============================================================
# 11. CLASS DISTRIBUTION
# ============================================================

class_distribution = (
    labeled_df["is_late"]
    .value_counts()
    .sort_index()
    .rename_axis("is_late")
    .reset_index(name="count")
)

class_distribution["label"] = (
    class_distribution["is_late"]
    .map({
        0: "on_time_or_early",
        1: "late",
    })
)

class_distribution["percentage"] = (
    class_distribution["count"]
    / len(labeled_df)
    * 100
)

class_distribution = class_distribution[
    [
        "is_late",
        "label",
        "count",
        "percentage",
    ]
]

print("\nCLASS DISTRIBUTION")
print("=" * 80)

display(class_distribution)


# ============================================================
# 12. CLASS IMBALANCE SUMMARY
# ============================================================

class_counts = (
    labeled_df["is_late"]
    .value_counts()
)

majority_count = class_counts.max()
minority_count = class_counts.min()

minority_percentage = (
    minority_count
    / len(labeled_df)
    * 100
)

imbalance_ratio = (
    majority_count
    / minority_count
)

print("\nCLASS IMBALANCE")
print("=" * 80)

print(
    f"Majority class: "
    f"{majority_count:,} orders"
)

print(
    f"Minority class: "
    f"{minority_count:,} orders"
)

print(
    f"Minority percentage: "
    f"{minority_percentage:.2f}%"
)

print(
    f"Majority / minority ratio: "
    f"{imbalance_ratio:.2f}:1"
)

# Project heuristic only.
# There is no universal class-imbalance cutoff.
if minority_percentage < 20:
    print(
        "\nDecision: Treat this dataset as class-imbalanced."
    )
    print(
        "Reason: the minority class represents less than "
        "20% of eligible orders."
    )
    print(
        "Note: 20% is a project heuristic, not a universal rule."
    )

else:
    print(
        "\nDecision: No strong class imbalance detected "
        "under the project heuristic."
    )


# ============================================================
# 13. IMPORTANT LEAKAGE NOTE
# ============================================================

print("\nDATA LEAKAGE NOTE")
print("=" * 80)

print(
    "order_delivered_customer_date and delivery_delay_days "
    "are used to construct/check the target."
)

print(
    "They must NOT be used as prediction-time features "
    "in the final ML model."
)


# ============================================================
# 14. SAVE LABELED ARTIFACT
# ============================================================

labeled_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

assert OUTPUT_PATH.exists()
assert OUTPUT_PATH.stat().st_size > 0


# ============================================================
# 15. VERIFY SAVED ARTIFACT
# ============================================================

saved_sample = pd.read_csv(
    OUTPUT_PATH,
    nrows=5,
)

print("\nARTIFACT CHECK")
print("=" * 80)

print(f"Saved artifact: {OUTPUT_PATH}")

print(
    f"Artifact size: "
    f"{OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB"
)

display(saved_sample)


# ============================================================
# 16. FINAL RESULT
# ============================================================

print("\n")
print("=" * 80)
print("NOTEBOOK 2 RESULT")
print("=" * 80)

print("SUCCESS")
print("Delivery label was created.")
print("Real orders were checked.")
print("Class distribution was measured.")
print("Class imbalance was evaluated.")
print(
    f"Labeled orders: {len(labeled_df):,}"
)
print(
    f"Artifact saved to: {OUTPUT_PATH}"
)
